# 09h Is the personal map the person or the session? (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Development analysis, not a prespecified test. Each training image was recorded in two of the four sessions, twice in each. The release does not store
sessions, but the authors shuffled repetitions with a fixed seed, so each repetition's session can be reconstructed.
Step 1 verifies that on the data: repetitions from the same session must agree more than repetitions from different
sessions. Step 2, on 4,000 images: four single-repetition maps per person (two per session), and the similarity of
their channel-mixing part within a session, across sessions and between people, per 100 ms window.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09h-h4bsessions', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"people": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10"], "k_images": 4000, "image_seed": 13, "map_window_ms": 100, "window_s": [0.1, 0.2], "analysis_windows": [3, 4], "n_perm": 1000, "verify_min_people": 8, "stable_ratio": 0.8, "specific_ratio": 0.5}')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

if not te.attached(C.EEG_RAW):
    raise RuntimeError("attach cross-subject-brain-encoding-eeg (built by 09g); this notebook does not download")
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
people, K = FIXED["people"], FIXED["k_images"]
N_TRAIN, N_TEST = len(z_train), len(z_test)
img = np.sort(np.random.default_rng(FIXED["image_seed"]).permutation(N_TRAIN)[:K])
lab_test, lab_train = te.session_labels("test"), te.session_labels("training")
query = np.concatenate([z_train[img], z_test])
pred_img, pred_test, reps, bio_test, verify = {}, {}, {}, {}, []
for s in people:
    tr = te.load_part(te.subject_file(C.EEG_RAW, s, "training"))
    ts = te.load_part(te.subject_file(C.EEG_RAW, s, "test"))
    times, bio_test[s] = ts["times"], ts["data"]
    vt = te.session_agreement(te.rep_correlations(ts["data"]), lab_test, n_perm=FIXED["n_perm"])
    vr = te.session_agreement(te.rep_correlations(tr["data"][img]), lab_train, n_perm=FIXED["n_perm"])
    verify.append({"person": s, "test_difference": vt["difference"], "test_p": vt["p"],
                    "training_difference": vr["difference"], "training_rank": vr["rank"]})
    # each person's own full-data encoder (the paper's OLS on 4-repetition averages), as in 09a-09c
    pred = pl.ols_paper(z_train, tr["data"].mean(1).reshape(N_TRAIN, -1), query)
    pred_img[s], pred_test[s] = pred[:K], pred[K:]
    reps[s] = tr["data"][img]  # (K, 4, 17, 100) single repetitions
    del tr, ts
    print(s, verify[-1], "|", f"{(time.monotonic() - T0) / 60:.1f} min")
vf = pd.DataFrame(verify)
vf.to_csv(C.METRICS_DIR / "09h_verification.csv", index=False)
VERIFIED = bool((vf.test_difference > 0).all() and (vf.test_p < 0.05).sum() >= FIXED["verify_min_people"]
                and (vf.training_rank == 1).sum() >= FIXED["verify_min_people"])
print("session labels verified:", VERIFIED)
print(vf.round(4).to_string())

In [ ]:
win = ha.window_index(times, FIXED["map_window_ms"])
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
off = ~np.eye(17, dtype=bool)
cos = lambda a, b: float(a.ravel() @ b.ravel() / (np.linalg.norm(a) * np.linalg.norm(b)))
E, L = np.where(lab_train == 0)[0], np.where(lab_train == 1)[0]
names = {"E1": E[0], "E2": E[1], "L1": L[0], "L2": L[1]}
maps, score_rows = {}, []
if VERIFIED:
    for i, s in enumerate(people):
        others = [q for q in people if q != s]
        a_k = np.mean([pred_img[q] for q in others], 0).reshape(K, 17, 100)
        a_t = np.mean([pred_test[q] for q in others], 0).reshape(N_TEST, 17, 100)
        maps[s] = {n: h4b.fit_map(a_k, reps[s][:, r], win, seed=13)[0] for n, r in names.items()}
        preds = {"avg": a_t, **{f"map_{n}": h4b.apply_map(maps[s][n], a_t, win) for n in ("E1", "L1")}}
        scored = paper_correlation(preds, bio_test[s], n_iter=100)
        for n, curve in scored["correlation"].items():
            row = {"person": s, "method": n, "r_100_200ms": float(curve[:, window].mean())}
            row.update({f"r_w{w}": float(curve[:, win == w].mean()) for w in np.unique(win)})
            score_rows.append(row)
        print(s, "maps fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")
    sim_rows = []
    for s in people:
        for w in sorted(maps[s]["E1"]):
            m = {n: maps[s][n][w] for n in names}
            for part, sel in (("full", np.ones((17, 17), bool)), ("offdiag", off)):
                same = np.mean([cos(m["E1"][sel], m["E2"][sel]), cos(m["L1"][sel], m["L2"][sel])])
                cross = np.mean([cos(m[a][sel], m[b][sel]) for a in ("E1", "E2") for b in ("L1", "L2")])
                between = np.mean([cos(m[n][sel], maps[q][n][w][sel]) for q in people if q != s for n in names])
                sim_rows.append({"person": s, "window": int(w), "part": part, "same_session": same,
                                 "cross_session": cross, "between_people": between})
    sim = pd.DataFrame(sim_rows)
    sim.to_csv(C.METRICS_DIR / "09h_similarity.csv", index=False)
    pd.DataFrame(score_rows).to_csv(C.METRICS_DIR / "09h_scores.csv", index=False)
    np.savez_compressed(C.METRICS_DIR / "09h_maps.npz", people=np.array(people), names=np.array(list(names)),
                        maps=np.stack([np.stack([np.stack([maps[s][n][w] for w in sorted(maps[s][n])]) for n in names]) for s in people]))

In [ ]:
summary = {"verified": VERIFIED, "verification": vf.to_dict(orient="records"), "k_images": K,
           "labels": {"training": lab_train.tolist(), "test_counts": np.bincount(lab_test).tolist()}}
if VERIFIED:
    a = sim[(sim.part == "offdiag") & sim.window.isin(FIXED["analysis_windows"])].groupby("person")[["same_session", "cross_session", "between_people"]].mean()
    ratio = a.cross_session / a.same_session
    t_cb = stats.ttest_1samp(a.cross_session - a.between_people, 0, alternative="greater")
    t_sc = stats.ttest_1samp(a.same_session - a.cross_session, 0)
    mean_ratio = float(a.cross_session.mean() / a.same_session.mean())
    verdict = ("person-stable" if mean_ratio >= FIXED["stable_ratio"] and (a.cross_session > a.between_people).all()
               else "session-specific" if mean_ratio <= FIXED["specific_ratio"] else "mixed")
    sc = pd.DataFrame(score_rows)
    gains = {n: float((sc[sc.method == n].set_index("person").r_100_200ms - sc[sc.method == "avg"].set_index("person").r_100_200ms).mean()) for n in ("map_E1", "map_L1")}
    summary.update({"offdiag_100_300ms": {"same_session": float(a.same_session.mean()), "cross_session": float(a.cross_session.mean()),
                                          "between_people": float(a.between_people.mean()), "cross_over_same": mean_ratio,
                                          "cross_minus_between_p": float(t_cb.pvalue), "same_minus_cross": float((a.same_session - a.cross_session).mean()),
                                          "same_minus_cross_p_two_sided": float(t_sc.pvalue),
                                          "cross_above_between_people": int((a.cross_session > a.between_people).sum()),
                                          "per_person_ratio": ratio.round(3).to_dict()},
                    "by_window": sim.groupby(["part", "window"])[["same_session", "cross_session", "between_people"]].mean().reset_index().to_dict(orient="records"),
                    "single_rep_map_gain_over_avg": gains, "verdict": verdict,
                    "rule": "person-stable if cross/same >= 0.8 and cross > between for every person; session-specific if <= 0.5; else mixed"})
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    ax[0].bar(range(len(vf)), vf.test_difference, color=["C2" if p < 0.05 else "C3" for p in vf.test_p])
    ax[0].set(xticks=range(len(vf)), title="Labels check: same- minus different-session\nrepetition agreement (test; green p < 0.05)")
    ax[0].set_xticklabels(vf.person, rotation=90, fontsize=7)
    g = sim[sim.part == "offdiag"].groupby("window")[["same_session", "cross_session", "between_people"]].mean()
    for col, lab in (("same_session", "same person, same session"), ("cross_session", "same person, other session"), ("between_people", "different people")):
        ax[1].plot(-150 + 100 * g.index, g[col], marker="o", label=lab)
    ax[1].axvline(0, color="grey", lw=0.5)
    ax[1].set(xlabel="window centre (ms)", ylabel="cosine, channel-mixing part", title="Single-repetition maps")
    ax[1].legend(fontsize=7)
    ax[2].scatter(a.same_session, a.cross_session)
    lim = [min(a.min()), max(a.max())]
    ax[2].plot(lim, lim, "k--", lw=0.6)
    ax[2].set(xlabel="same session", ylabel="other session", title="Per person, 100-300 ms")
    fig.tight_layout()
    fig.savefig(C.METRICS_DIR / "09h_figure.png", dpi=150)
    print(json.dumps({k: summary[k] for k in ("verdict", "offdiag_100_300ms", "single_rep_map_gain_over_avg")}, indent=1))
(C.METRICS_DIR / "09h_summary.json").write_text(json.dumps(summary, indent=2, default=float))

In [ ]:
files = [C.METRICS_DIR / "09h_summary.json", C.METRICS_DIR / "09h_verification.csv"]
if VERIFIED:
    files += [C.METRICS_DIR / n for n in ("09h_similarity.csv", "09h_scores.csv", "09h_maps.npz", "09h_figure.png")]
export_to_dataset(slug="cross-subject-brain-encoding-09h-outputs", title="XSUB 09h H4b sessions outputs",
                  files=files, include_dirs=[], message="h4b maps across sessions")